# 01 — Ingestion EDA (NSE Bhavcopy)

Spot-check Phase-1 ingestion: series mix, coverage gaps, symbol normalization, and corporate-action factor sanity.

**Rules reminder:** EQ/BE only · adjusted prices for returns · India $R_f$ (RBI 91d / MIBOR).

In [ ]:
from pathlib import Path
import polars as pl

from src.data.paths import RAW_BHAVCOPY_DIR, PROCESSED_BHAVCOPY_DIR, ensure_data_dirs
from src.data.bhavcopy import load_bhavcopy_dir, write_bhavcopy_parquet, ALLOWED_SERIES
from src.data.symbols import SymbolMap
from src.data.risk_free import example_rbi_tbill_template, load_risk_free_csv

ensure_data_dirs()
RAW_BHAVCOPY_DIR, PROCESSED_BHAVCOPY_DIR

## Load raw bhavcopy archives

Place `cm*bhav.csv.zip` and/or `BhavCopy_NSE_CM_*.csv.zip` files under `data/raw/bhavcopy/`, then run:

In [ ]:
panel = load_bhavcopy_dir(RAW_BHAVCOPY_DIR)
print(f"rows={panel.height:,}  symbols={panel['symbol'].n_unique() if panel.height else 0}")
panel.head()

In [ ]:
if panel.height:
    series_mix = (
        panel.group_by("series")
        .len()
        .sort("len", descending=True)
    )
    assert set(panel["series"].unique().to_list()) <= ALLOWED_SERIES
    display(series_mix)

    coverage = (
        panel.group_by("trade_date")
        .agg(pl.col("symbol").n_unique().alias("n_symbols"))
        .sort("trade_date")
    )
    display(coverage.describe())
else:
    print("No archives found — download with src.data.bhavcopy.download_bhavcopy_range(...)")

## Persist year-partitioned Parquet + risk-free template

In [ ]:
if panel.height:
    written = write_bhavcopy_parquet(panel)
    print("wrote", written)

rf_template = example_rbi_tbill_template()
print("RBI T-Bill template:", rf_template)
load_risk_free_csv(rf_template).head()

## Symbol normalization smoke check

In [ ]:
sm = SymbolMap()
print(sm.to_yfinance("RELIANCE"), sm.normalize_nse("infy.ns"), sm.normalize_nse("HDFC"))